In [1]:
import pandas as pd

In [2]:
FF_df= pd.read_csv("Historical Fpts by Game 2020.csv")

In [3]:
#making sure the dataset transferred over well
FF_df.head()

,week,Oppt,DKP,Price,FDP,Price_f,YHP,Price_y,Name,Team,Pos
0,1,nyj,33.18,6500.0,28.18,7900.0,28.18,31.0,Josh Allen,Buffalo Bills,QB
1,2,@ mia,37.48,6700.0,34.48,8200.0,34.48,35.0,Josh Allen,Buffalo Bills,QB
2,3,lar,36.24,6900.0,32.24,8100.0,32.24,36.0,Josh Allen,Buffalo Bills,QB
3,4,@ lvr,25.42,7300.0,25.42,8600.0,25.42,40.0,Josh Allen,Buffalo Bills,QB
4,5,@ ten,18.32,7500.0,18.32,8400.0,18.32,37.0,Josh Allen,Buffalo Bills,QB


In [4]:
#filtering for columns that will be used
DraftKing_df=FF_df[["Name", "Team", "Pos", "week", "DKP", "Price"]]

In [5]:
#ensuring last code ran well
DraftKing_df.head()

,Name,Team,Pos,week,DKP,Price
0,Josh Allen,Buffalo Bills,QB,1,33.18,6500.0
1,Josh Allen,Buffalo Bills,QB,2,37.48,6700.0
2,Josh Allen,Buffalo Bills,QB,3,36.24,6900.0
3,Josh Allen,Buffalo Bills,QB,4,25.42,7300.0
4,Josh Allen,Buffalo Bills,QB,5,18.32,7500.0


In [6]:
#Using Week 11 for reasons mentioned in README.
Week_Snip = DraftKing_df[DraftKing_df["week"]<=11]

In [7]:
#ensuring last code ran well
Week_Snip["week"].max()

11

In [8]:
#Here I am going to isolate week 12 from the original data frame. This would not be considered data leakage because a user would know the current price of a player before they buy.
Week_Twelve= FF_df[FF_df["week"]==12]

In [9]:
#Just checking everything worked as intended
Week_Twelve.head()

,week,Oppt,DKP,Price,FDP,Price_f,YHP,Price_y,Name,Team,Pos
10,12,lac,17.48,7600.0,16.48,8600.0,16.48,39.0,Josh Allen,Buffalo Bills,QB
26,12,@ det,36.12,7400.0,33.12,8700.0,33.12,37.0,Deshaun Watson,Houston Texans,QB
42,12,@ tam,35.28,8000.0,31.28,9000.0,31.28,37.0,Patrick Mahomes II,Kansas City Chiefs,QB
58,12,@ nwe,8.90,8200.0,8.90,9100.0,8.90,39.0,Kyler Murray,Arizona Cardinals,QB
74,12,chi,25.64,6700.0,25.64,8600.0,25.64,33.0,Aaron Rodgers,Green Bay Packers,QB


In [10]:
Week_Twelve["week"].max()

12

In [11]:
Week_Twelve["week"].min()

12

In [12]:
#Now I want to find the average points scored by players from Weeks 1-11.
Avg_DKP=Week_Snip.groupby("Name")["DKP"].mean()

In [13]:
#Checking that the last code worked
Avg_DKP.head()

Name
A.J. Brown       17.2250
A.J. Green        7.6700
A.J. McCarron        NaN
AJ Dillon         2.0500
Aaron Jones      20.3125
Name: DKP, dtype: float64

In [14]:
#I want to make sure that the players I am evaluating actually have enough data to analyze. For reasons explained in the readme, I have chosen a minimum of 3 games.
Game_Count=Week_Snip.groupby("Name")["DKP"].count()

In [15]:
#Checking that the code worked
Game_Count.head()

Name
A.J. Brown        8
A.J. Green       10
A.J. McCarron     0
AJ Dillon         6
Aaron Jones       8
Name: DKP, dtype: int64

In [16]:
#This code will make sure that every player that gets their average calculated will have atleast played 3 games.
Qualified_Players=Game_Count[Game_Count>=3]

In [17]:
#A.J McCarron should be gone now.
Qualified_Players.head()

Name
A.J. Brown        8
A.J. Green       10
AJ Dillon         6
Aaron Jones       8
Aaron Rodgers    10
Name: DKP, dtype: int64

In [18]:
#Now the players that show up will have atleast played 3 games between weeks 1-11
Clean_Avg=Avg_DKP[Avg_DKP.index.isin(Qualified_Players.index)]

In [19]:
#No more NaN!
Clean_Avg.head()

Name
A.J. Brown       17.2250
A.J. Green        7.6700
AJ Dillon         2.0500
Aaron Jones      20.3125
Aaron Rodgers    25.4760
Name: DKP, dtype: float64

In [20]:
#As of now this is a series, but in order to merge this with Week_Twelve I will need to make this into a data frame
Clean_Avg=Clean_Avg.to_frame()

In [21]:
Clean_Avg.head()

,DKP
Name,
A.J. Brown,17.2250
A.J. Green,7.6700
AJ Dillon,2.0500
Aaron Jones,20.3125
Aaron Rodgers,25.4760


In [22]:
#Name is still being stored as an index. This would not allow for a successful merge. This code will clean this up and allow for a merge.
Clean_Avg=Clean_Avg.reset_index()

In [23]:
Clean_Avg.head()

,Name,DKP
0,A.J. Brown,17.2250
1,A.J. Green,7.6700
2,AJ Dillon,2.0500
3,Aaron Jones,20.3125
4,Aaron Rodgers,25.4760


In [24]:
User_View = Clean_Avg.merge(Week_Twelve,on= "Name")

In [25]:
#Now we managed to merge average DKP (its being shown as DKP_x, because Week_Twelve already had a column named DKP)
User_View.head()

,Name,DKP_x,week,Oppt,DKP_y,Price,FDP,Price_f,YHP,Price_y,Team,Pos
0,A.J. Brown,17.2250,12,@ ind,25.80,6700.0,23.80,7500.0,23.80,25.0,Tennessee Titans,WR
1,A.J. Green,7.6700,12,nyg,0.00,3600.0,. 0,5300.0,. 0,14.0,Cincinnati Bengals,WR
2,AJ Dillon,2.0500,12,chi,NaN,4000.0,.,4900.0,.,10.0,Green Bay Packers,RB
3,Aaron Jones,20.3125,12,chi,10.00,7000.0,9.50,8200.0,9.50,31.0,Green Bay Packers,RB
4,Aaron Rodgers,25.4760,12,chi,25.64,6700.0,25.64,8600.0,25.64,33.0,Green Bay Packers,QB


In [26]:
#Removing data from other websites and unavaveraged DKP from week 12
User_View=User_View.drop(columns= ["DKP_y","FDP","Price_f","YHP","Price_y"])

In [27]:
User_View.head()

,Name,DKP_x,week,Oppt,Price,Team,Pos
0,A.J. Brown,17.2250,12,@ ind,6700.0,Tennessee Titans,WR
1,A.J. Green,7.6700,12,nyg,3600.0,Cincinnati Bengals,WR
2,AJ Dillon,2.0500,12,chi,4000.0,Green Bay Packers,RB
3,Aaron Jones,20.3125,12,chi,7000.0,Green Bay Packers,RB
4,Aaron Rodgers,25.4760,12,chi,6700.0,Green Bay Packers,QB


In [28]:
User_View=User_View.rename(columns={"DKP_x":"AVG_DKP"})

In [29]:
#Renamed column
User_View.head()

,Name,AVG_DKP,week,Oppt,Price,Team,Pos
0,A.J. Brown,17.2250,12,@ ind,6700.0,Tennessee Titans,WR
1,A.J. Green,7.6700,12,nyg,3600.0,Cincinnati Bengals,WR
2,AJ Dillon,2.0500,12,chi,4000.0,Green Bay Packers,RB
3,Aaron Jones,20.3125,12,chi,7000.0,Green Bay Packers,RB
4,Aaron Rodgers,25.4760,12,chi,6700.0,Green Bay Packers,QB


In [30]:
#Adding a column PPD(Points per Dollar), that calculates how many points a player will give your per 1000 dollars based on thier season average.
User_View["PPD"] = User_View['AVG_DKP']/User_View['Price']*1000

In [31]:
User_View.head()

,Name,AVG_DKP,week,Oppt,Price,Team,Pos,PPD
0,A.J. Brown,17.2250,12,@ ind,6700.0,Tennessee Titans,WR,2.570896
1,A.J. Green,7.6700,12,nyg,3600.0,Cincinnati Bengals,WR,2.130556
2,AJ Dillon,2.0500,12,chi,4000.0,Green Bay Packers,RB,0.512500
3,Aaron Jones,20.3125,12,chi,7000.0,Green Bay Packers,RB,2.901786
4,Aaron Rodgers,25.4760,12,chi,6700.0,Green Bay Packers,QB,3.802388


In [32]:
User_View.sort_values(by= "PPD")

,Name,AVG_DKP,week,Oppt,Price,Team,Pos,PPD
151,Dwayne Harris,-0.333333,12,@ gnb,3000.0,Chicago Bears,WR,-0.111111
106,Dante Pettis,-0.333333,12,@ cin,3000.0,New York Giants,WR,-0.111111
366,Robert Griffin III,-0.180000,12,@ pit,4100.0,Baltimore Ravens,QB,-0.043902
400,Tim Boyle,-0.120000,12,chi,4100.0,Green Bay Packers,QB,-0.029268
304,Mason Rudolph,-0.113333,12,bal,4100.0,Pittsburgh Steelers,QB,-0.027642
...,...,...,...,...,...,...,...,...
240,Jordan Howard,5.600000,12,sea,NaN,Philadelphia Eagles,RB,NaN
246,Josh Malone,2.266667,12,mia,NaN,New York Jets,WR,NaN
363,Rico Gafford,0.000000,12,@ atl,NaN,Las Vegas Raiders,WR,NaN
384,Seth Roberts,2.366667,12,@ min,NaN,Carolina Panthers,WR,NaN


In [33]:
#Players that have no price for week twelve weren't playable and will not be accounted for.
User_View=User_View[User_View["Price"].notna()]

In [34]:
User_View.head()

,Name,AVG_DKP,week,Oppt,Price,Team,Pos,PPD
0,A.J. Brown,17.2250,12,@ ind,6700.0,Tennessee Titans,WR,2.570896
1,A.J. Green,7.6700,12,nyg,3600.0,Cincinnati Bengals,WR,2.130556
2,AJ Dillon,2.0500,12,chi,4000.0,Green Bay Packers,RB,0.512500
3,Aaron Jones,20.3125,12,chi,7000.0,Green Bay Packers,RB,2.901786
4,Aaron Rodgers,25.4760,12,chi,6700.0,Green Bay Packers,QB,3.802388


In [35]:
#Thought negative DKP players got removed by the NaN filter. Just wanted to make sure everything was still there since a glance at the previous code is not proof.
User_View[User_View["AVG_DKP"]<0]

,Name,AVG_DKP,week,Oppt,Price,Team,Pos,PPD
106,Dante Pettis,-0.333333,12,@ cin,3000.0,New York Giants,WR,-0.111111
151,Dwayne Harris,-0.333333,12,@ gnb,3000.0,Chicago Bears,WR,-0.111111
304,Mason Rudolph,-0.113333,12,bal,4100.0,Pittsburgh Steelers,QB,-0.027642
366,Robert Griffin III,-0.180000,12,@ pit,4100.0,Baltimore Ravens,QB,-0.043902
400,Tim Boyle,-0.120000,12,chi,4100.0,Green Bay Packers,QB,-0.029268


In [36]:
#Want PPD to be in descending order
User_View=User_View.sort_values(by="PPD", ascending=False)

In [37]:
#It worked! Now I can make a ranking of the top 25 players based of PPD.
User_View.head()

,Name,AVG_DKP,week,Oppt,Price,Team,Pos,PPD
96,Dak Prescott,30.328000,12,was,4000.0,Dallas Cowboys,QB,7.582000
341,Odell Beckham Jr.,12.402857,12,@ jac,3000.0,Cleveland Browns,WR,4.134286
373,Russell Wilson,28.614000,12,@ phi,7500.0,Seattle Seahawks,QB,3.815200
4,Aaron Rodgers,25.476000,12,chi,6700.0,Green Bay Packers,QB,3.802388
164,Gardner Minshew,21.200000,12,cle,5600.0,Jacksonville Jaguars,QB,3.785714


In [38]:
#Boom! It's done now.
User_View.head(25)

,Name,AVG_DKP,week,Oppt,Price,Team,Pos,PPD
96,Dak Prescott,30.328000,12,was,4000.0,Dallas Cowboys,QB,7.582000
341,Odell Beckham Jr.,12.402857,12,@ jac,3000.0,Cleveland Browns,WR,4.134286
373,Russell Wilson,28.614000,12,@ phi,7500.0,Seattle Seahawks,QB,3.815200
4,Aaron Rodgers,25.476000,12,chi,6700.0,Green Bay Packers,QB,3.802388
164,Gardner Minshew,21.200000,12,cle,5600.0,Jacksonville Jaguars,QB,3.785714
340,O.J. Howard,9.400000,12,kan,2500.0,Tampa Bay Buccaneers,TE,3.760000
252,Justin Herbert,26.740000,12,@ buf,7200.0,Los Angeles Chargers,QB,3.713889
278,Kyler Murray,30.266000,12,@ nwe,8200.0,Arizona Cardinals,QB,3.690976
374,Ryan Fitzpatrick,20.025714,12,@ nyj,5500.0,Miami Dolphins,QB,3.641039
378,Ryan Tannehill,21.030000,12,@ ind,5800.0,Tennessee Titans,QB,3.625862
